# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one (client_hash_id, content_hash_id) pair, evaluated at a decision date. Features are computed from a past window ending at the decision date; the label comes from a future window starting right after it. I'm using month=2026-03 (data_14.parquet) as my mid-panel development month, and reserving the _sample table (June 2026) as a sealed test month.

In [81]:
import duckdb
con = duckdb.connect()

# Client-level filter: clients both are active and has access to at least one of search/analytics
clients = con.execute("""
    SELECT client_hash_id, access_profile, is_active, gsc_data_start, ga4_data_start
    FROM read_parquet('../../data/raw/dim_clients.parquet')
    WHERE is_active IS TRUE
      AND access_profile != 'no_search_or_analytics_access'
""").df()
print("usable clients:", len(clients))

# Content-level filter: undeleted and published content
content_meta = con.execute("""
    SELECT client_hash_id, content_hash_id, content_type, is_published, is_deleted,
           content_created_date
    FROM read_parquet('../../data/raw/dim_content.parquet')
    WHERE is_deleted IS FALSE AND is_published IS TRUE
""").df()
print("usable content rows:", len(content_meta))

DEV_FILE = "../../data/raw/fact_content_daily_performance/data_14.parquet"

grain_check = con.execute(f"""
    SELECT client_hash_id, content_hash_id, report_date, COUNT(*) AS n
    FROM read_parquet('{DEV_FILE}')
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
""").df()
print("duplicate (client, content, date) rows:", len(grain_check))
assert len(grain_check) == 0, "grain assumption violated - investigate before proceeding"

usable clients: 56
usable content rows: 411540
duplicate (client, content, date) rows: 0


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

I joined fact_content_daily_performance with dim_clients and dim_content to see the full field set before bucketing. Feature columns are the six time-varying GSC/GA4 metrics. Context columns are IDs plus the fields I need for filtering, not modeling: access_profile and the two *_data_start dates tell me whether a client-window pair is genuinely measured or just structurally missing; is_published/is_deleted tell me whether a content row should exist in the label space at all. I excluded everything content-shape-related (word count, provider, keyword metadata) because they describe the page itself, not its traffic trajectory, and belong to a different lane (archetype/refresh scoring).

In [82]:
display(con.execute("SELECT * FROM read_parquet('../../data/raw/dim_content.parquet') LIMIT 5").df())
display(con.execute("SELECT * FROM read_parquet('../../data/raw/dim_clients.parquet') LIMIT 5").df())

FEATURE_COLS = [
    "gsc_impressions", "gsc_clicks", "gsc_avg_position",
    "ga4_sessions", "ga4_engaged_sessions", "sessions_organic",
]

CONTEXT_COLS = [
    "client_hash_id", "content_hash_id", "report_date",   # from fact table
    "access_profile", "gsc_data_start", "ga4_data_start",  # from dim_clients — for filtering + censoring control
    "content_type", "is_published", "is_deleted",          # from dim_content — for filtering
]

EXCLUDED_COLS = [
    "sessions_ai", "ai_chatgpt", "ai_perplexity", "ai_gemini", "ai_copilot", "ai_claude", "ai_meta", "ai_other",
    "provider_used", "model_used", "char_count", "word_count",
    "keyword_char_count", "keyword_token_count", "url_char_count", "category_count",
    "last_optimized_date", "optimization_eligible_date",
]

exclusion_reasons = {
    "sessions_ai (+ per-model columns)": "too sparse per (client, content) to support a class label; EDA-only",
    "provider_used, model_used": "describes how content was generated, not how it performs; not a time-varying growth signal",
    "char_count, word_count, keyword_*, url_char_count, category_count": "static content-shape metadata; belongs to a content-quality/archetype lane, not momentum prediction — noted but not used here",
    "last_optimized_date, optimization_eligible_date": "belongs to the 'refresh scoring' lane, not this lane's target definition",
    "fact_content_query_90d.parquet": "different grain (client, content, query, 90d) than my (client, content, day) grain",

    "content never observed in any of the 18 months (~9,476 items)": "no baseline traffic in any window - growth/decline cannot be defined without a reference point; excluded before feature computation, not zero-filled",
    "content with zero rows in the dev month but observed elsewhere (~156,331 items)": "kept via LEFT JOIN + zero-fill on click/impression/session sums; this is a real 'no traffic this window' signal, not missingness - confirmed via full 18-month presence check",
    "GSC: 25778 rows (8.2%) dropped ":"11 of 56 usable clients (19.6%) had not started GSC tracking by the end of this 15-day dev window (gsc_data_start ranging 2026-03-25 to 2026-06-02). This reflects real, ongoing client onboarding, not a data quality defect.",
    
}

for k, v in exclusion_reasons.items():
    print(f"- {k}: {v}")

,client_hash_id,content_hash_id,keyword_hash_id,url_hash_id,keyword_char_count,keyword_token_count,url_char_count,content_created_date,content_updated_date,content_type,...,category_count,keyword_created_date,provider_used,model_used,char_count,word_count,last_optimized_date,optimization_eligible_date,is_published,is_deleted
0,client_04660893ae39614a,content_004de9653278b5a4,keyword_e754999ab88dd9f2,url_d6091f18cf628794,22,4,108,2026-05-30,2026-07-01,keyword article,...,3,2026-05-12,gemini-generate-content,gemini-3-flash-preview,15682,2555,NaT,NaT,True,False
1,client_04660893ae39614a,content_00dc5efae381b2ab,keyword_4329d7aede8e208b,url_3a66d2f2e36823ca,31,6,95,2026-06-12,2026-07-01,keyword article,...,4,2026-06-01,gemini-generate-content,gemini-3-flash-preview,15438,2430,NaT,NaT,True,False
2,client_04660893ae39614a,content_01410f2556c327ac,keyword_9b08047d3d2a0406,url_809eda7a7e20b3b2,22,5,82,2026-05-09,2026-07-01,keyword article,...,4,2026-05-06,gemini-generate-content,gemini-3-flash-preview,16576,2645,NaT,NaT,True,False
3,client_04660893ae39614a,content_019f27f634053ca7,keyword_e7cec7ab1804c1c2,url_5fb42bafc4399861,14,3,92,2026-06-15,2026-06-15,keyword article,...,4,2026-06-01,gemini-generate-content,gemini-3-flash-preview,15457,2522,NaT,NaT,True,False
4,client_04660893ae39614a,content_01efa71faea45dcc,keyword_56b0062a1d8b7524,url_ece0abc3e5fb75f9,24,6,98,2026-05-21,2026-06-01,keyword article,...,4,2026-05-12,gemini-generate-content,gemini-3-flash-preview,15776,2552,NaT,NaT,True,False


,client_hash_id,is_active,has_gsc_access,has_ga4_access,access_profile,client_created_date,client_updated_date,gsc_data_start,ga4_data_start
0,client_04660893ae39614a,True,True,True,gsc_and_ga4,2026-04-15,2026-06-27,NaT,2026-05-22
1,client_05475c07ed21a83a,True,False,False,no_search_or_analytics_access,2026-04-01,2026-06-27,NaT,NaT
2,client_06d356715a8ff3b6,True,True,True,gsc_and_ga4,2026-03-23,2026-07-05,2026-04-10,2026-04-06
3,client_0797ff3a1fc9a6a5,True,False,False,no_search_or_analytics_access,2025-05-26,2026-06-27,2025-11-05,NaT
4,client_08a6a72ff48e62c0,True,True,False,gsc_only,2025-05-26,2026-06-27,2025-09-24,NaT


- sessions_ai (+ per-model columns): too sparse per (client, content) to support a class label; EDA-only
- provider_used, model_used: describes how content was generated, not how it performs; not a time-varying growth signal
- char_count, word_count, keyword_*, url_char_count, category_count: static content-shape metadata; belongs to a content-quality/archetype lane, not momentum prediction — noted but not used here
- last_optimized_date, optimization_eligible_date: belongs to the 'refresh scoring' lane, not this lane's target definition
- fact_content_query_90d.parquet: different grain (client, content, query, 90d) than my (client, content, day) grain
- content never observed in any of the 18 months (~9,476 items): no baseline traffic in any window - growth/decline cannot be defined without a reference point; excluded before feature computation, not zero-filled
- content with zero rows in the dev month but observed elsewhere (~156,331 items): kept via LEFT JOIN + zero-fill on click/im

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [83]:
DEV_FILE = "../../data/raw/fact_content_daily_performance/data_14.parquet"  # fix according to the verification in index 0
TEST_FILE = "../../data/raw/fact_content_daily_performance_sample.parquet"

print("usable clients:", len(clients))
print("usable content rows:", len(content_meta))
print("duplicate (client, content, date) rows:", len(grain_check))

slice_stats = con.execute(f"""
    SELECT
        COUNT(*)                        AS total_rows,
        COUNT(DISTINCT client_hash_id)  AS n_clients,
        COUNT(DISTINCT content_hash_id) AS n_content,
        MIN(report_date)                AS first_date,
        MAX(report_date)                AS last_date
    FROM read_parquet('{DEV_FILE}')
""").df()
display(slice_stats)

DECISION_DATE = "2026-03-15"  # end of the dev window, used to censor clients whose GSC start date is after this point
# Query 3: availability, combining row-level IS TRUE flags with client-level onboarding status.
# Row-level flags alone understate onboarding gaps: a client with no GSC tracking yet
# usually has NO ROWS AT ALL for that period (nothing to backfill), so a plain
# "report_date < gsc_data_start" row count misses most of the effect. The client-level
# CTE captures the real onboarding picture instead: how many usable clients had not
# started GSC tracking by the end of this dev window, INCLUDING clients whose
# gsc_data_start is NULL despite claiming GSC access (a metadata inconsistency,
# treated conservatively as censored since the start date cannot be trusted).
availability = con.execute(f"""
    WITH row_level AS (
        SELECT
            COUNT(*) AS total_rows,
            COUNT(*) FILTER (WHERE f.gsc_data_available IS TRUE) AS gsc_available_rows,
            COUNT(*) FILTER (WHERE f.ga4_data_available IS TRUE) AS ga4_available_rows,
            COUNT(*) FILTER (WHERE c.access_profile = 'no_search_or_analytics_access') AS rows_no_access_client,
            COUNT(*) FILTER (WHERE f.report_date < c.gsc_data_start) AS rows_before_gsc_start,
            COUNT(*) FILTER (WHERE f.report_date < c.ga4_data_start) AS rows_before_ga4_start
        FROM read_parquet('{DEV_FILE}') f
        LEFT JOIN read_parquet('../../data/raw/dim_clients.parquet') c
          ON f.client_hash_id = c.client_hash_id
    ),
    client_level AS (
        SELECT
            COUNT(*) AS n_usable_clients,
            COUNT(*) FILTER (WHERE access_profile = 'ga4_only') AS n_gsc_structurally_absent,
            COUNT(*) FILTER (
                WHERE access_profile != 'ga4_only' AND gsc_data_start IS NULL
            ) AS n_gsc_start_unknown,
            COUNT(*) FILTER (WHERE gsc_data_start >= DATE '{DECISION_DATE}') AS n_gsc_future_start,
            COUNT(*) FILTER (
                WHERE access_profile = 'ga4_only'
                OR gsc_data_start IS NULL
                OR gsc_data_start >= DATE '{DECISION_DATE}'
            ) AS n_gsc_censored_or_unknown
        FROM read_parquet('../../data/raw/dim_clients.parquet')
        WHERE is_active IS TRUE AND access_profile != 'no_search_or_analytics_access'
    )
    SELECT * FROM row_level, client_level
""").df()
display(availability)


usable clients: 56
usable content rows: 411540
duplicate (client, content, date) rows: 0


,total_rows,n_clients,n_content,first_date,last_date
0,9841378,55,331437,2026-03-01,2026-03-31


,total_rows,gsc_available_rows,ga4_available_rows,rows_no_access_client,rows_before_gsc_start,rows_before_ga4_start,n_usable_clients,n_gsc_structurally_absent,n_gsc_start_unknown,n_gsc_future_start,n_gsc_censored_or_unknown
0,9841378,3611061,413966,8401,4604,1133759,56,1,6,11,18


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Limitation 1 — client onboarding is ongoing across the 18-month span. A fixed calendar
decision date does not work equally well for all clients: of the 56 usable clients,
a meaningful share had not started GSC tracking by the end of this 15-day dev window
(see the query below for the exact count and rate). This is not fixable by picking a
different fixed date — some clients will always postdate any single cutoff. The real
modeling pipeline (Week 5+) should compute decision dates relative to each client's own
tracking start, or restrict training rows to (client, decision-date) pairs where
sufficient history already exists.

Limitation 2 — Of the 56 usable clients, 18 (32.1%) have no usable GSC start date for this dev window: 11 have a confirmed future start date, 1 is structurally GSC-absent (access_profile == 'ga4_only'), and 6 claim GSC access but have no recorded start date — a genuine metadata inconsistency, not an edge case. All three groups are treated the same way downstream: conservatively excluded, since a zero cannot be trusted without a known measurement start.

Note on two different measurements: the row-level check below (rows_before_gsc_start,
computed over the full month) is much smaller than the client-level onboarding rate,
because a client with no GSC tracking yet typically has no rows at all for that period —
there is nothing to compare dates against. The client-level count is the more meaningful
number for understanding how much of the client base this actually affects.

In [84]:
no_access_rows = availability["rows_no_access_client"][0]
total = availability["total_rows"][0]

n_usable_clients = availability["n_usable_clients"][0]
n_gsc_structurally_absent = availability["n_gsc_structurally_absent"][0]
n_gsc_start_unknown = availability["n_gsc_start_unknown"][0]
n_gsc_future_start = availability["n_gsc_future_start"][0]
n_gsc_censored_or_unknown = availability["n_gsc_censored_or_unknown"][0]

print(f"rows from clients with no access: {no_access_rows} ({no_access_rows/total:.1%})")
print()
print("GSC usability breakdown among the 56 usable clients:")
print(f"  structurally absent (access_profile == 'ga4_only'): {n_gsc_structurally_absent}")
print(f"  claimed GSC access but gsc_data_start is NULL (metadata inconsistency): {n_gsc_start_unknown}")
print(f"  gsc_data_start postdates this dev window (confirmed future onboarding): {n_gsc_future_start}")
print(f"  total unusable for GSC in this window: {n_gsc_censored_or_unknown} / {n_usable_clients} "
      f"({n_gsc_censored_or_unknown/n_usable_clients:.1%})")

rows from clients with no access: 8401 (0.1%)

GSC usability breakdown among the 56 usable clients:
  structurally absent (access_profile == 'ga4_only'): 1
  claimed GSC access but gsc_data_start is NULL (metadata inconsistency): 6
  gsc_data_start postdates this dev window (confirmed future onboarding): 11
  total unusable for GSC in this window: 18 / 56 (32.1%)


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.